# PyTorch 训练循环练习

本模块把张量、自动求导、损失函数和优化器连成完整训练流程。目标是训练模型学习近似关系 $y=3x+2$。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## 练习 1：准备回归数据 ⭐

创建 100 个从 -1 到 1 的输入点，形状为 `(100, 1)`；目标值为 `3*x+2`，再加入标准差约为 0.1 的随机噪声。

In [ ]:
# TODO
x = None
y = None

assert x.shape == (100, 1)
assert y.shape == (100, 1)
assert torch.allclose(x[[0, -1], 0], torch.tensor([-1.0, 1.0]))
assert (y - (3 * x + 2)).std() < 0.2
print("✅ 练习 1 通过")

## 练习 2：模型、损失函数和优化器 ⭐⭐

建立一个输入和输出维数均为 1 的线性层；使用均方误差损失与 SGD 优化器，学习率设为 0.05。

In [ ]:
# TODO
model = None
loss_fn = None
optimizer = None

assert isinstance(model, nn.Linear)
assert model.in_features == 1 and model.out_features == 1
assert isinstance(loss_fn, nn.MSELoss)
assert isinstance(optimizer, torch.optim.SGD)
print("✅ 练习 2 通过")

## 练习 3：完成一次训练步骤 ⭐⭐

把模型参数先清零，然后依次执行：清梯度、前向计算、计算损失、反向传播、更新参数。

In [ ]:
one_step_model = nn.Linear(1, 1)
with torch.no_grad():
    one_step_model.weight.zero_()
    one_step_model.bias.zero_()
one_step_optimizer = torch.optim.SGD(one_step_model.parameters(), lr=0.05)

before_weight = one_step_model.weight.detach().clone()

# TODO
one_step_optimizer.zero_grad()
prediction = None
step_loss = None
# 反向传播
# 更新参数

after_weight = one_step_model.weight.detach().clone()
assert step_loss.ndim == 0
assert one_step_model.weight.grad is not None
assert not torch.equal(before_weight, after_weight)
print("✅ 练习 3 通过")

## 练习 4：使用 DataLoader 分批训练 ⭐⭐⭐

创建批大小为 16、会打乱数据的 DataLoader。训练一个新的线性模型 120 个 epoch，并把每个 epoch 的平均损失加入 `loss_history`。

In [ ]:
dataset = TensorDataset(x, y)

# TODO
dataloader = None
trained_model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(trained_model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
loss_history = []

# TODO：编写 120 个 epoch 的训练循环

learned_weight = trained_model.weight.item()
learned_bias = trained_model.bias.item()
assert len(loss_history) == 120
assert loss_history[-1] < loss_history[0]
assert abs(learned_weight - 3.0) < 0.2
assert abs(learned_bias - 2.0) < 0.2
print(f"✅ 练习 4 通过：w={learned_weight:.3f}, b={learned_bias:.3f}")

## 练习 5：训练模式与评估模式 ⭐⭐

Dropout 在训练时随机丢弃元素，在评估时保持确定。分别生成两个训练输出和两个评估输出。

In [ ]:
dropout_model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(p=0.5))
input_batch = torch.ones(32, 4)

# TODO
# 切换到训练模式后得到 train_output_1 和 train_output_2
train_output_1 = None
train_output_2 = None
# 切换到评估模式，并在不记录梯度时得到 eval_output_1 和 eval_output_2
eval_output_1 = None
eval_output_2 = None

assert not torch.equal(train_output_1, train_output_2)
assert torch.equal(eval_output_1, eval_output_2)
assert not eval_output_1.requires_grad
print("✅ 练习 5 通过")

## 练习 6：编写可复用的训练函数 ⭐⭐⭐

补全 `train_one_epoch`。它需要切换训练模式、移动数据、清梯度、计算损失、反向传播和更新参数，最后返回按样本数加权的平均损失。

In [ ]:
def train_one_epoch(model, dataloader, loss_fn, optimizer, device):
    # TODO
    pass


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
function_model = nn.Linear(1, 1).to(device)
function_optimizer = torch.optim.SGD(function_model.parameters(), lr=0.05)
average_loss = train_one_epoch(function_model, dataloader, loss_fn, function_optimizer, device)

assert isinstance(average_loss, float)
assert average_loss >= 0
assert next(function_model.parameters()).device.type == device.type
print(f"✅ 练习 6 通过，设备={device}，平均损失={average_loss:.4f}")

## 过关标准

你应该能不看答案写出训练循环的固定顺序，并理解 `zero_grad → forward → loss → backward → step` 中每一步的作用。